# 01 · Spatial contracts and a tiny city

**Question:** Can we preserve coordinates, units and feature identities across a twin pipeline?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Ask Astra to detect a swapped coordinate pair, explain local metres versus geographic degrees, and propose validation rules.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Coordinate frame and editable inputs
GeoJSON uses `[longitude, latitude]`. Calculations use a local equirectangular approximation in metres over a 2 km area.
This is not an EPSG projected CRS; use a vetted projection library for larger areas or survey work.
The site-specific bounding box is deliberately stronger than global coordinate range checks.


In [ ]:
c = city()
population_total = 1200
lon, lat = lonlat(*c['xy'].T)
x2,y2 = local_xy(lon,lat)
error_m = np.max(np.hypot(x2-c['xy'][:,0],y2-c['xy'][:,1]))
assert error_m < 1e-6
assert ((lon>25.8)&(lon<26.1)&(lat>-24.8)&(lat<-24.5)).all()
features_geo = []
for i,(x,y) in enumerate(c['xy']):
    w,d=c['width'][i]/2,c['depth'][i]/2
    ring=[list(map(float,lonlat(a,b))) for a,b in [(x-w,y-d),(x+w,y-d),(x+w,y+d),(x-w,y+d),(x-w,y-d)]]
    features_geo.append(dict(type='Feature',id=f'b{i:03d}',properties=dict(synthetic=True,floors=int(c['floors'][i])),geometry=dict(type='Polygon',coordinates=[ring])))
geojson=dict(type='FeatureCollection',features=features_geo)
assert len({f['id'] for f in features_geo})==len(features_geo)
print('Buildings:',len(features_geo),'| round-trip error (m):',error_m)


In [ ]:
fig,ax=plt.subplots()
for f in features_geo:
    ring=np.array(f['geometry']['coordinates'][0]); xx,yy=local_xy(*ring.T)
    ax.fill(xx,yy,color='#087f8c',alpha=.55)
map_axes(ax,'Synthetic urban footprint inventory'); plt.show()
export_json('01_buildings.geojson',geojson)


## Deliberately break the contract
A pair can pass world-wide latitude/longitude bounds and still be in the wrong country.
Spatial plausibility needs study-area bounds and provenance, not only valid JSON.


In [ ]:
good=[float(lon[0]),float(lat[0])]; bad=good[::-1]
def in_study_area(pair):
    return 25.8<pair[0]<26.1 and -24.8<pair[1]<-24.5
assert in_study_area(good) and not in_study_area(bad)
print('Astra prompt: Audit these GeoJSON coordinate pairs near Gaborone:',good,bad,
      'Explain which checks fail. Do not invent a surveyed coordinate.')


## Teaching lab: Audit coordinates
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 01_buildings.geojson plus the two coordinate pairs printed above. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Identify coordinate order, frame and units. Explain which pair violates the study bounds and why global bounds alone are insufficient.

**Implementation brief — review the runnable extension below:**
> Add a GeoJSON audit returning feature IDs for unclosed rings, duplicate IDs and points outside the study bounds. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** An intentionally reversed coordinate pair fails; unchanged valid features pass; no automatic axis swapping.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Audit a real footprint extract before measuring it

Load 125 small OpenStreetMap building ways, retaining ODbL attribution and source timestamps. Audit them before calculating approximate local areas. A second copy deliberately contains four errors: swapped axes, an open ring, a crossing ring and a duplicate ID. The audit supports simple polygon exteriors; unsupported geometry is flagged rather than silently flattened.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Audit a real footprint extract before measuring it', ['OSM rings + source IDs', 'Bounds, closure, crossings', 'Flag map + valid areas', 'Incomplete map ≠ census'])
plt.show()


In [ ]:
from copy import deepcopy
from extension_tools import audit_footprints, load_data
ex_real=load_data('gaborone_buildings.geojson')
ex_audit=audit_footprints(ex_real,bbox=[25.89,-24.70,25.94,-24.65])
ex_good=[r for r in ex_audit if not r['issues']]
assert ex_good and len({f['id'] for f in ex_real['features']})==len(ex_real['features'])
ex_broken=deepcopy(ex_real['features'][:1])*1
ex_axis=deepcopy(ex_real['features'][0]); ex_axis['id']='axis_error'
ex_axis['geometry']['coordinates'][0]=[p[::-1] for p in ex_axis['geometry']['coordinates'][0]]
ex_open=deepcopy(ex_real['features'][0]); ex_open['id']='open_ring'; ex_open['geometry']['coordinates'][0].pop()
ex_cross=dict(type='Feature',id='crossed',properties={},geometry=dict(type='Polygon',coordinates=[[[25.91,-24.68],[25.912,-24.678],[25.91,-24.678],[25.912,-24.68],[25.91,-24.68]]]))
ex_broken += [ex_axis,ex_open,ex_cross,deepcopy(ex_real['features'][0])]
ex_faults=audit_footprints(dict(features=ex_broken),[25.89,-24.70,25.94,-24.65])
assert sum(bool(r['issues']) for r in ex_faults)==4
fig,axes=plt.subplots(1,3,figsize=(14,4))
for feature,audit in zip(ex_real['features'],ex_audit):
    ring=np.array(feature['geometry']['coordinates'][0]); x,y=local_xy(*ring.T)
    axes[0].fill(x,y,color='#087f8c' if not audit['issues'] else '#c44e52',alpha=.65)
axes[0].set(title='Real OSM footprints: audit first',xlabel='Local east (m)',ylabel='Local north (m)',aspect='equal')
axes[1].hist([r['area_m2'] for r in ex_good],bins=15,color='#087f8c'); axes[1].set(title='Only valid rings contribute area',xlabel='Approximate footprint m²',ylabel='Buildings')
ex_labels=['axis_error','open_ring','crossed',ex_real['features'][0]['id']]
axes[2].barh(ex_labels,[len(r['issues']) for r in ex_faults[1:]],color='#c44e52'); axes[2].set(title='Injected defects detected',xlabel='Failed checks')
fig.text(.01,.01,'© OpenStreetMap contributors · ODbL 1.0 · openstreetmap.org/copyright',fontsize=9)
fig.tight_layout(rect=[0,.05,1,1]); extension_figure=fig
extension_evidence=dict(provenance=ex_real['provenance'],features=ex_broken,study_bbox=[25.89,-24.70,25.94,-24.65])
extension_expected=dict(invalid_features=4)
extension_task='Audit these simple polygon records for coordinate order, study bounds, closure, crossings and duplicate IDs. Return invalid_features as the number of records with any defect.'
print('Real extract:',len(ex_real['features']),'| accepted:',len(ex_good),'| injected bad records:',4)


In [ ]:
extension_prompt,extension_reference=export_investigation('01',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 01_extension_prompt.json and optionally 01_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `01_extension_prompt.json` and `01_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('01_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
